### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [3]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")


### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10), # trigger summarization after 10-length messages
            keep=("messages",4) # keep the last 4 messages after summarization
        )
    ]
)


In [7]:
config = {"configurable": {"thread_id":"test_1"}}

In [9]:
questions=[
    "What is the capital of France?",
    "What is the population of France?",
    "What is the largest city in France?",
    "What is the official language of France?",
    "What is the currency of France?",
    "What is the main religion in France?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(f"Messages:{response['messages']}")
    print(f"Messages length:{len(response['messages'])}")

Messages:[HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='42c0ad9e-614e-4e5c-8e52-0d2938ce6af7'), AIMessage(content='<think>\nOkay, so the user is asking, "What is the capital of France?" Hmm, I need to make sure I answer this correctly. Let me think. I remember from school that France is a country in Europe, and its capital is a well-known city. Let me recall... Paris? Yeah, I think that\'s right. But wait, maybe I should double-check to be sure. Sometimes capitals can be easily confused with other major cities.\n\nLet me think about other cities in France. There\'s Lyon, Marseille, Toulouse, Bordeaux... None of those are the capital. The capital is usually the political center. Paris is where the government is located, right? The Eiffel Tower is in Paris, and it\'s a major cultural hub. Also, the French president\'s residence is in Paris, the Élysée Palace. That makes sense. I don\'t think any other French city has that status.\n

### Token Size LIMIT

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",500), # trigger summarization after 550 tokens
            keep=("tokens",200), # keep the last 200 tokens after summarization
        ),
    ]
)

config_1 = {"configurable": {"thread_id": "test-2"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [13]:
cities = ["Paris", "London", "Tokyo"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config_1
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~137 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='280d340f-c1f4-4055-921a-f7a0c1fbf52f'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The description says it returns a long response to use more tokens. Since the user wants hotels in Paris, I need to call this function with the city set to Paris. I should make sure the arguments are correctly formatted as JSON. Alright, the required parameter is city, and the type is string. So the tool call should have the name "search_hotels" and the arguments with "city": "Paris". Let me double-check for any typos. Everything looks good. Now, I\'ll structure the tool_call XML with the correct JSON inside.\n', 'tool_calls': [{'id': '2v436gy2g', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_ho

### Fraction LIMIT Based on LLM Context Window Size

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% of the context window
            keep=("fraction", 0.002),     # 0.2% of the context window
        ),
    ],
)

config_2 = {"configurable": {"thread_id": "test-3"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config_2
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 131072  # qwen3:32b context window
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~68 tokens (0.0519%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='0bf91eed-2d4d-42d5-9383-f8e9edb98cf9'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the available tools. There's a function called search_hotels that requires a city parameter. The user mentioned Paris, so I need to call that function with the city set to Paris. I should make sure the parameters are correctly formatted as JSON. Let me structure the tool call accordingly.\n", 'tool_calls': [{'id': '1tfbet0xh', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 147, 'total_tokens': 243, 'completion_time': 0.136431451, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.005841305, 'prompt_tokens_details': None, 'queue_time': 0.045931504, 'total_time':